[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/03_data_analysis_and_eda/03_univariate_and_bivariate_analysis.ipynb)

# Bài 03 — Phân tích đơn biến, hai biến và đa biến

**🎯 Mục tiêu**
- Chọn **đúng loại biểu đồ + thống kê** cho từng kiểu biến: số – số, số – phân loại, phân loại – phân loại.
- Nhận diện phân phối lệch, outlier, category hiếm; biết khi nào cần log-transform.
- Đo mức độ liên hệ: Pearson/Spearman, so sánh nhóm, **chi-square + Cramér's V**.
- Phân tích đa biến: tô màu theo biến thứ ba, phát hiện tương tác.

Dữ liệu: `house_prices.csv` (mục tiêu `price_billion_vnd` — regression) và `customer_churn.csv` (mục tiêu `churn` — classification).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/03_data_analysis_and_eda

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from ml_course.visualization import set_style

set_style()
houses = pd.read_csv("../../data/sample/house_prices.csv")
churn = pd.read_csv("../../data/sample/customer_churn.csv")
houses.head()

## 1. Bảng chọn biểu đồ

| Kiểu phân tích | Biểu đồ | Thống kê |
|---|---|---|
| 1 biến số | histogram, boxplot, KDE | mean, median, std, IQR, skewness |
| 1 biến phân loại | bar chart (tần suất) | value_counts, số category, category hiếm |
| số – số | scatter (hexbin nếu nhiều điểm) | Pearson, Spearman |
| số – phân loại | boxplot / violin theo nhóm | trung bình/median theo nhóm, ANOVA |
| phân loại – phân loại | crosstab, stacked bar | chi-square, Cramér's V |
| đa biến | scatter tô màu/kích thước, scatter matrix, heatmap | tương quan, tương tác |

## 2. Phân tích đơn biến — biến số

**Độ lệch (skewness):** $> 0$ lệch phải (đuôi dài bên phải: giá, thu nhập), $< 0$ lệch trái. Dữ liệu lệch mạnh làm mean bị kéo lệch, ảnh hưởng tới các model giả định phân phối chuẩn (linear regression) → thường thử **log-transform**.

In [ ]:
price = houses["price_billion_vnd"]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
axes[0].hist(price, bins=40)
axes[0].axvline(price.mean(), color="red", label=f"mean = {price.mean():.2f}")
axes[0].axvline(price.median(), color="k", ls="--", label=f"median = {price.median():.2f}")
axes[0].legend()
axes[0].set_title(f"Giá nhà — skew = {price.skew():.2f}")
axes[1].hist(np.log(price), bins=40, color="#16a34a")
axes[1].set_title(f"log(giá) — skew = {np.log(price).skew():.2f}")
axes[2].boxplot(price, vert=False)
axes[2].set_title("Boxplot: điểm ngoài râu = ứng viên outlier")
plt.show()

**Đọc boxplot:** hộp = $Q_1$ đến $Q_3$ (50% dữ liệu ở giữa), vạch giữa = median, râu kéo tới điểm xa nhất trong $[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$, điểm bên ngoài = ứng viên outlier. *Ứng viên* — với dữ liệu lệch phải, nhiều điểm "outlier" thật ra là giá trị hợp lệ (nhà rất đắt)!

In [ ]:
def describe_numeric(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return pd.Series({
        "mean": s.mean(), "median": s.median(), "std": s.std(), "min": s.min(), "max": s.max(),
        "skew": s.skew(), "IQR": iqr, "n_outliers_IQR": ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum(),
    })


houses.select_dtypes("number").apply(describe_numeric).T.round(2)

## 3. Phân tích đơn biến — biến phân loại

Quan tâm: số category (cardinality), category **hiếm** (ít mẫu → thống kê không ổn định, model dễ overfit), mất cân bằng giữa các nhóm.

In [ ]:
vc = houses["district"].value_counts()
print(vc)
print("\nTỉ lệ:", (vc / len(houses)).round(3).to_dict())

## 4. Hai biến số — scatter, Pearson, Spearman

- **Pearson** đo quan hệ **tuyến tính**.
- **Spearman** = Pearson trên **thứ hạng** → đo quan hệ **đơn điệu** (tăng/giảm, không nhất thiết tuyến tính), ít nhạy với outlier.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, col in zip(axes, ["area_m2", "age_years"]):
    r = stats.pearsonr(houses[col], price).statistic
    rho = stats.spearmanr(houses[col], price).statistic
    ax.scatter(houses[col], price, s=8, alpha=0.4)
    ax.set_xlabel(col)
    ax.set_ylabel("giá (tỷ VND)")
    ax.set_title(f"Pearson = {r:.2f} | Spearman = {rho:.2f}")
plt.show()

Khi có hàng chục nghìn điểm, scatter bị "đè" thành một mảng đen → dùng **hexbin** (đếm mật độ) hoặc giảm `alpha`:

In [ ]:
fraud = pd.read_csv("../../data/sample/fraud_transactions.csv")
fig, ax = plt.subplots(figsize=(7, 4))
hb = ax.hexbin(fraud["distance_from_home_km"], np.log10(fraud["amount_vnd"]), gridsize=40, cmap="viridis", bins="log")
fig.colorbar(hb, label="log10(số giao dịch)")
ax.set_xlabel("khoảng cách từ nhà (km)")
ax.set_ylabel("log10(số tiền)")
ax.set_title(f"Hexbin với {len(fraud):,} giao dịch")
plt.show()

## 5. Số – phân loại: so sánh phân phối giữa các nhóm

In [ ]:
order = houses.groupby("district")["price_billion_vnd"].median().sort_values().index
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([houses.loc[houses["district"] == d, "price_billion_vnd"] for d in order], tick_labels=order)
ax.set_ylabel("giá (tỷ VND)")
ax.set_title("Giá theo quận (sắp xếp theo median)")
plt.show()
houses.groupby("district")["price_billion_vnd"].agg(["count", "median", "mean", "std"]).loc[order].round(2)

**Cẩn thận với kết luận:** Quận 1 đắt hơn — nhưng có thể một phần vì nhà ở Quận 1 có diện tích khác? Phân tích hai biến chỉ cho thấy **liên hệ**, chưa tách được ảnh hưởng của các biến khác. Giá **trên mỗi m²** là so sánh công bằng hơn:

In [ ]:
houses["price_per_m2_million"] = houses["price_billion_vnd"] * 1000 / houses["area_m2"]
houses.groupby("district")["price_per_m2_million"].median().loc[order].round(1)

## 6. Phân loại – phân loại: crosstab, chi-square, Cramér's V

**Crosstab** chuẩn hoá theo hàng cho tỉ lệ churn của từng nhóm. **Chi-square test** kiểm định hai biến có độc lập không; **Cramér's V** ∈ [0, 1] đo **mức độ** liên hệ (p-value chỉ nói "có liên hệ", không nói "mạnh hay yếu", và với dữ liệu lớn p-value hầu như luôn rất nhỏ).

In [ ]:
ct = pd.crosstab(churn["contract"], churn["churn"])
print(ct, "\n")
print(pd.crosstab(churn["contract"], churn["churn"], normalize="index").round(3))


def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2, p, _, _ = stats.chi2_contingency(table)
    n = table.to_numpy().sum()
    return np.sqrt(chi2 / (n * (min(table.shape) - 1))), p


for col in ["contract", "internet_service", "payment_method", "senior"]:
    v, p = cramers_v(churn[col], churn["churn"])
    print(f"{col:18s} Cramér's V = {v:.3f}   p-value = {p:.1e}")

In [ ]:
ct_norm = pd.crosstab(churn["internet_service"], churn["churn"], normalize="index")
ax = ct_norm.plot(kind="bar", stacked=True, figsize=(6, 3.5), color=["#2563eb", "#dc2626"], rot=0)
ax.set_ylabel("tỉ lệ")
ax.legend(["ở lại", "rời bỏ"], loc="upper right")
ax.set_title("Stacked bar: tỉ lệ churn theo loại internet")
plt.show()

## 7. Đa biến — tô màu theo biến thứ ba, tìm tương tác

Hai biến có thể **tương tác**: ảnh hưởng của biến này phụ thuộc vào giá trị của biến kia. Ví dụ: tenure ảnh hưởng tới churn **mạnh hơn** với hợp đồng tháng.

In [ ]:
churn["tenure_group"] = pd.cut(churn["tenure_months"], [-1, 6, 12, 24, 48, 72])
rates = churn.pivot_table(values="churn", index="tenure_group", columns="contract", aggfunc="mean", observed=True)
ax = rates.plot(marker="o", figsize=(8, 4))
ax.set_ylabel("tỉ lệ churn")
ax.set_title("Tương tác tenure × contract: các đường KHÔNG song song")
plt.show()
rates.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(houses["area_m2"], houses["price_billion_vnd"], c=houses["distance_to_center_km"],
                s=10 + 20 * houses["has_garden"], cmap="plasma_r", alpha=0.6)
fig.colorbar(sc, label="khoảng cách tới trung tâm (km)")
ax.set_xlabel("diện tích (m²)")
ax.set_ylabel("giá (tỷ VND)")
ax.set_title("4 biến trên 1 hình: x, y, màu = khoảng cách, kích thước = có vườn")
plt.show()

**Đọc hình:** cùng diện tích, nhà gần trung tâm (màu tím đậm) đắt hơn hẳn → khoảng cách là biến quan trọng, và quan hệ diện tích–giá có **nhiều "dải"** (theo quận). Model tuyến tính đơn giản chỉ với diện tích sẽ bỏ lỡ cấu trúc này.

## 8. Mẹo trình bày

- Luôn ghi **số lượng mẫu** khi so sánh tỉ lệ giữa các nhóm (nhóm 5 người có churn 80% chẳng nói lên điều gì).
- Sắp xếp category theo giá trị (median, tỉ lệ) thay vì theo bảng chữ cái.
- Dữ liệu lệch → trục log hoặc log-transform.
- Một biểu đồ = một thông điệp; đặt tiêu đề là **kết luận** ("Hợp đồng tháng churn cao gấp 3") thay vì mô tả ("Churn theo contract").

## 🧠 Kiểm tra nhanh

1. Quan hệ giữa hai biến tăng đơn điệu nhưng cong (vd: $y = e^x$). Pearson hay Spearman phản ánh tốt hơn?
2. Chi-square cho p-value = 1e−30 với 1 triệu mẫu. Có nghĩa là hai biến liên hệ **mạnh** không?
3. Vì sao so sánh giá **trên m²** giữa các quận công bằng hơn so sánh giá nhà?
4. Thế nào là tương tác giữa hai feature? Cho một ví dụ.

## 📌 Tóm tắt

- Mỗi cặp kiểu biến có biểu đồ và thống kê phù hợp (bảng ở mục 1).
- Lệch phải → log; boxplot cho ứng viên outlier — kiểm tra trước khi xoá.
- Pearson (tuyến tính) vs Spearman (đơn điệu); Cramér's V đo độ mạnh liên hệ phân loại.
- Phân tích hai biến chỉ cho thấy liên hệ; cần đa biến để thấy tương tác và loại trừ biến gây nhiễu.